In [1]:
%pip install ultralytics opencv-python 

  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached polars-1.43.2-py3-none-any.whl.metadata (11 kB)
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.1.6-py3-none-any.whl.metadata (13 kB)
  Using cached polars_runtime_32-1.43.2-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ----------------------------- ---------- 1.0/1.4 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 6.0 MB/s  0:00:00
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
Using cached nvidia_ml_py-13.610.43-py3-none-any.whl (53 kB)
Using cached polars-1.43.2

In [25]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [26]:
from ultralytics import YOLO,solutions
import cv2
import numpy as np
model = YOLO("yolo26n-pose.pt")

In [45]:
import time
import cv2
import numpy as np
from ultralytics import YOLO
from scipy.signal import butter, sosfilt, sosfilt_zi

# COCO pose indices
LEFT_ARM = [5, 7, 9]          # shoulder, elbow, wrist
RIGHT_ARM = [6, 8, 10]
LEFT_LINE = (5, 11, 15)       # shoulder, hip, ankle
RIGHT_LINE = (6, 12, 16)

CONF_MIN = 0.4
STRAIGHT_TOL = 15.0           # hip angle near 180 = good posture
UP_ANGLE = 145.0              # elbow angle for "up"
DOWN_ANGLE = 90.0             # elbow angle for "down"
CUTOFF_HZ = 5.0
FILTER_ORDER = 2
SIDE_VOTES_NEEDED = 20
SIDE_MARGIN = 0.10


def pt(p):
    return (int(p[0]), int(p[1]))


def joint_angle(a, b, c):
    """Angle ABC in degrees, or None if points are too close."""
    ba = a[:2] - b[:2]
    bc = c[:2] - b[:2]
    na = np.linalg.norm(ba)
    nc = np.linalg.norm(bc)
    if na < 1e-6 or nc < 1e-6:
        return None
    cos = np.clip(np.dot(ba, bc) / (na * nc), -1.0, 1.0)
    return float(np.degrees(np.arccos(cos)))


def guess_side(kpts):
    """Use elbow/wrist confidence: the nearer side is usually more visible."""
    left = (kpts[7][2] + kpts[9][2]) / 2
    right = (kpts[8][2] + kpts[10][2]) / 2
    if abs(right - left) < SIDE_MARGIN:
        return None
    return "right" if right > left else "left"


def make_smoother(fps):
    """Causal Butterworth. Returns (sos, zi). zi starts as None."""
    if fps is None or fps < 1:
        fps = 30.0
    nyquist = 0.5 * fps
    cutoff = min(CUTOFF_HZ, 0.9 * nyquist)
    sos = butter(FILTER_ORDER, cutoff / nyquist, btype="low", output="sos")
    return sos, None


def smooth_xy(xy, sos, zi):
    """Smooth 17x2 xy coords. Returns (smoothed_xy, new_zi)."""
    flat = xy.reshape(-1).astype(np.float64)
    if zi is None:
        zi = sosfilt_zi(sos)[:, :, None] * flat[None, None, :]
    out, zi = sosfilt(sos, flat[None, :], axis=0, zi=zi)
    return out[0].reshape(17, 2), zi


def count_rep(elbow, stage, reps):
    """AIGym-style up/down crossing. Returns (new_stage, new_reps)."""
    if elbow is None:
        return stage, reps
    if elbow < DOWN_ANGLE:
        if stage == "up":
            reps += 1
        stage = "down"
    elif elbow > UP_ANGLE:
        stage = "up"
    return stage, reps


def draw_posture(frame, kpts, chain):
    """Draw shoulder-ankle line. Returns a short status string."""
    i_sh, i_hip, i_ank = chain
    if min(kpts[i_sh][2], kpts[i_hip][2], kpts[i_ank][2]) < CONF_MIN:
        return "low conf"

    sh, hip, ank = kpts[i_sh][:2], kpts[i_hip][:2], kpts[i_ank][:2]
    angle = joint_angle(sh, hip, ank)
    if angle is None:
        return "low conf"

    good = angle >= 180.0 - STRAIGHT_TOL
    color = (0, 255, 0) if good else (0, 0, 255)

    ac = ank - sh
    t = np.clip(np.dot(hip - sh, ac) / max(np.dot(ac, ac), 1e-6), 0.0, 1.0)
    on_line = sh + t * ac

    if good:
        label = f"GOOD {angle:.0f}deg"
    elif hip[1] > on_line[1]:
        label = f"hips sagging {angle:.0f}deg"
    else:
        label = f"hips too high {angle:.0f}deg"

    cv2.line(frame, pt(sh), pt(ank), color, 4)
    cv2.line(frame, pt(sh), pt(hip), (255, 255, 0), 2)
    cv2.line(frame, pt(hip), pt(ank), (255, 255, 0), 2)
    cv2.line(frame, pt(hip), pt(on_line), (0, 255, 255), 2)
    for p in (sh, hip, ank):
        cv2.circle(frame, pt(p), 8, color, -1)

    cv2.putText(frame, label, (pt(sh)[0], pt(sh)[1] - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
    return label


def draw_arm(frame, kpts, arm, elbow):
    sh, el, wr = kpts[arm[0]][:2], kpts[arm[1]][:2], kpts[arm[2]][:2]
    cv2.line(frame, pt(sh), pt(el), (255, 0, 255), 3)
    cv2.line(frame, pt(el), pt(wr), (255, 0, 255), 3)
    for p in (sh, el, wr):
        cv2.circle(frame, pt(p), 6, (255, 0, 255), -1)
    if elbow is not None:
        cv2.putText(frame, f"{elbow:.0f}deg", (pt(el)[0] + 12, pt(el)[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2)


def draw_hud(frame, reps, stage, side, status):
    cv2.putText(frame, f"REPS: {reps}", (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 255), 3)
    cv2.putText(frame, f"stage: {stage}   side: {side or 'detecting...'}", (20, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 220, 220), 2)
    if status:
        cv2.putText(frame, status, (20, 125),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 220, 220), 2)

In [47]:
USE_WEBCAM = False
VIDEO_PATH = r"C:\Users\arika\Documents\pose_estimation\pushup5.mp4"
WEBCAM_ID = 0
SAVE_OUTPUT = True
OUTPUT_PATH = "Pushups.demo.video.output.avi"

source = WEBCAM_ID if USE_WEBCAM else VIDEO_PATH

if USE_WEBCAM:
    cap = cv2.VideoCapture(source, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
else:
    cap = cv2.VideoCapture(source)

assert cap.isOpened(), f"Could not open {source}"

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
if fps is None or fps < 1:
    fps = 30.0

print(f"{'webcam' if USE_WEBCAM else 'video'}: {w}x{h} @ {fps:.1f} fps")

model = YOLO("yolo26n-pose.pt")

writer = None
if SAVE_OUTPUT:
    writer = cv2.VideoWriter(
        OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
    )

WINDOW = "Pushups"
cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
try:
    from ctypes import windll
    sw, sh = windll.user32.GetSystemMetrics(0), windll.user32.GetSystemMetrics(1)
except Exception:
    sw, sh = 1920, 1080
scale = min((sw - 80) / max(w, 1), (sh - 120) / max(h, 1), 1.0)
cv2.resizeWindow(WINDOW, max(1, int(w * scale)), max(1, int(h * scale)))

# live state
left_votes = 0
right_votes = 0
side = None
arm = None
chain = None
sos, zi = make_smoother(fps)
stage = "-"
reps = 0
status = ""
next_frame_time = time.perf_counter()
frame_delay = 1.0 / fps

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        result = model.track(frame, persist=True, verbose=False)[0]
        has_person = result.keypoints is not None and len(result.keypoints.data) > 0

        if has_person:
            kpts = result.keypoints.data.cpu().numpy()[0].astype(np.float64)

            # lock side after enough confident votes, then keep it
            if side is None:
                guess = guess_side(kpts)
                if guess == "left":
                    left_votes += 1
                elif guess == "right":
                    right_votes += 1
                if left_votes + right_votes >= SIDE_VOTES_NEEDED:
                    side = "right" if right_votes > left_votes else "left"
                    arm = RIGHT_ARM if side == "right" else LEFT_ARM
                    chain = RIGHT_LINE if side == "right" else LEFT_LINE
                    print(f"using {side} side  (votes L={left_votes} R={right_votes})")

            xy, zi = smooth_xy(kpts[:, :2], sos, zi)
            kpts[:, :2] = xy

            if arm is not None:
                elbow = joint_angle(kpts[arm[0]], kpts[arm[1]], kpts[arm[2]])
                stage, reps = count_rep(elbow, stage, reps)
                draw_arm(frame, kpts, arm, elbow)
                status = draw_posture(frame, kpts, chain)
        else:
            status = "no person"

        draw_hud(frame, reps, stage, side, status)

        if writer is not None:
            writer.write(frame)
        cv2.imshow(WINDOW, frame)

        # webcam already comes in at camera speed; slow video files to real time
        if USE_WEBCAM:
            wait = 1
        else:
            next_frame_time += frame_delay
            wait = max(1, int((next_frame_time - time.perf_counter()) * 1000))
        if cv2.waitKey(wait) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    if writer is not None:
        writer.release()
    cv2.destroyAllWindows()

print(f"reps: {reps}")




video: 1280x720 @ 25.0 fps
using left side  (votes L=20 R=0)
reps: 4
